# Medical RAG UI

Run the cells from top to bottom. Upload reference files, click **Build RAG index**, enter a question, and click **Ask**. The answer is generated only from the retrieved document context. Set `USE_OPENAI=true`, `OPENAI_API_KEY`, and optionally `base_url`, `EMBEDDING_MODEL`, and `model` in `.env` before running the setup cell.

In [2]:
!pip install python-docx


[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import os
import sqlite3
from io import BytesIO
from pathlib import Path

import ipywidgets as widgets
from IPython.display import display, Markdown
from dotenv import load_dotenv
from docx import Document as WordDocument
from langchain_core.documents import Document
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from pypdf import PdfReader
from openai import APIError

load_dotenv()

use_openai = os.getenv("USE_OPENAI", "false").strip().lower() == "true"
if not use_openai:
    raise RuntimeError("Set USE_OPENAI=true in .env before running this cell.")

api_key = os.getenv("OPENAI_API_KEY")
if not api_key:
    raise RuntimeError("OPENAI_API_KEY is missing from .env.")

embedding_model = os.getenv("EMBEDDING_MODEL", "text-embedding-3-small")
model_name = os.getenv("model", "gpt-4o-mini")
base_url = os.getenv("base_url") or None
embeddings = OpenAIEmbeddings(model=embedding_model, api_key=api_key, base_url=base_url)
llm = ChatOpenAI(model=model_name, api_key=api_key, base_url=base_url, temperature=0)

DATA_DIR = Path("data")
PATIENT_REPORTS_DB = DATA_DIR / "patient_reports.db"
GENERATED_REPORTS_DB = DATA_DIR / "generated_reports.db"
DATA_DIR.mkdir(exist_ok=True)

seed_documents = [
    Document(page_content="LangChain 是一個快速串接語言模型與外部工具的開源框架。", metadata={"source": "built-in"}),
    Document(page_content="向量資料庫專門用於存取和檢索向量化資料，可實現語義搜尋。", metadata={"source": "built-in"}),
    Document(page_content="RAG 結合檢索與生成能力，提供有根據的即時回答。", metadata={"source": "built-in"}),
]
print(f"Ready: {model_name} with {embedding_model}")

Ready: gpt-5.6-terra with text-embedding-3-small


In [4]:
def uploaded_items(upload_value):
    return upload_value.values() if isinstance(upload_value, dict) else upload_value


def read_uploaded_file(uploaded_file):
    file_name = uploaded_file["name"]
    file_bytes = bytes(uploaded_file["content"])
    suffix = file_name.lower().rsplit(".", 1)[-1] if "." in file_name else ""
    if suffix in {"txt", "md", "csv", "json"}:
        return file_bytes.decode("utf-8", errors="replace")
    if suffix == "pdf":
        reader = PdfReader(BytesIO(file_bytes))
        return "\n".join(page.extract_text() or "" for page in reader.pages)
    if suffix == "docx":
        document = WordDocument(BytesIO(file_bytes))
        return "\n".join(paragraph.text for paragraph in document.paragraphs)
    raise ValueError(f"Unsupported file type: .{suffix or 'unknown'}")


def chunk_text(text, chunk_size=1200, overlap=200):
    normalized = " ".join(text.split())
    if not normalized:
        return []
    chunks = []
    start = 0
    while start < len(normalized):
        end = min(start + chunk_size, len(normalized))
        chunks.append(normalized[start:end])
        if end == len(normalized):
            break
        start = end - overlap
    return chunks


def ensure_sqlite_tables():
    with sqlite3.connect(PATIENT_REPORTS_DB) as connection:
        connection.execute("""CREATE TABLE IF NOT EXISTS patient_reports (
            id INTEGER PRIMARY KEY AUTOINCREMENT, patient_id TEXT NOT NULL,
            report_type TEXT, report_text TEXT NOT NULL, file_name TEXT,
            created_at TEXT NOT NULL DEFAULT CURRENT_TIMESTAMP)""")
    with sqlite3.connect(GENERATED_REPORTS_DB) as connection:
        connection.execute("""CREATE TABLE IF NOT EXISTS generated_reports (
            id INTEGER PRIMARY KEY AUTOINCREMENT, patient_report_id INTEGER NOT NULL,
            patient_id TEXT NOT NULL, generated_text TEXT NOT NULL, tone TEXT NOT NULL,
            model_name TEXT, created_at TEXT NOT NULL DEFAULT CURRENT_TIMESTAMP)""")


ensure_sqlite_tables()
documents = list(seed_documents)
vector_store = InMemoryVectorStore.from_documents(documents, embeddings)
last_answer = ""
last_sources = []

In [5]:
uploader = widgets.FileUpload(accept=".txt,.md,.csv,.json,.pdf,.docx", multiple=True, description="Choose files")
build_button = widgets.Button(description="Build RAG index", button_style="primary", icon="database")
question = widgets.Textarea(placeholder="Ask a question about the uploaded material...", layout=widgets.Layout(width="100%", height="80px"))
ask_button = widgets.Button(description="Ask", button_style="success", icon="comment")
clear_button = widgets.Button(description="Clear", icon="trash")
patient_id = widgets.Text(description="Patient ID:", placeholder="Optional")
save_button = widgets.Button(description="Save answer", icon="save")
status = widgets.HTML(value="<b>Status:</b> Ready with built-in documents.")
answer_output = widgets.Output()
sources_output = widgets.Output()
history_output = widgets.Output()


def set_status(message, error=False):
    color = "#b00020" if error else "#176b3a"
    status.value = f'<b>Status:</b> <span style="color:{color}">{message}</span>'


def rebuild_index(_=None):
    global documents, vector_store
    documents = list(seed_documents)
    skipped = []
    for uploaded_file in uploaded_items(uploader.value):
        try:
            text = read_uploaded_file(uploaded_file)
            documents.extend(Document(page_content=part, metadata={"source": uploaded_file["name"]}) for part in chunk_text(text))
        except (KeyError, TypeError, ValueError, UnicodeError) as error:
            skipped.append(str(error))
    try:
        vector_store = InMemoryVectorStore.from_documents(documents, embeddings)
    except (APIError, OSError, RuntimeError, ValueError) as error:
        set_status(f"Could not build index: {error}", error=True)
        return
    message = f"Indexed {len(documents)} chunks from {len(uploaded_items(uploader.value))} uploaded file(s)."
    if skipped:
        message += " Skipped: " + "; ".join(skipped)
    set_status(message, error=bool(skipped))


def ask_question(_=None):
    global last_answer, last_sources
    query = question.value.strip()
    if not query:
        set_status("Enter a question first.", error=True)
        return
    try:
        last_sources = vector_store.similarity_search(query, k=4)
        context = "\n\n".join(f"[{i}] {doc.page_content}" for i, doc in enumerate(last_sources, 1))
        prompt = ("Answer the user's question using only the context below. "
                  "If the context does not contain the answer, say so clearly. "
                  "Do not invent medical facts.\n\nContext:\n" + context + "\n\nQuestion: " + query)
        response = llm.invoke(prompt)
        last_answer = str(response.content)
    except (APIError, OSError, RuntimeError, ValueError) as error:
        set_status(f"Question failed: {error}", error=True)
        return
    with answer_output:
        answer_output.clear_output()
        display(Markdown(last_answer))
    with sources_output:
        sources_output.clear_output()
        for index, doc in enumerate(last_sources, 1):
            print(f"[{index}] {doc.metadata.get('source', 'unknown')}\n{doc.page_content}\n")
    set_status(f"Answered using {len(last_sources)} retrieved source chunk(s).")


def clear_ui(_=None):
    global last_answer, last_sources
    question.value = ""
    last_answer, last_sources = "", []
    answer_output.clear_output()
    sources_output.clear_output()
    set_status("Ready.")


def save_answer(_=None):
    if not patient_id.value.strip() or not last_answer:
        set_status("Enter a patient ID and ask a question before saving.", error=True)
        return
    with sqlite3.connect(PATIENT_REPORTS_DB) as connection:
        report_id = connection.execute(
            "INSERT INTO patient_reports (patient_id, report_type, report_text, file_name) VALUES (?, ?, ?, ?)",
            (patient_id.value.strip(), "RAG question", question.value.strip(), None),
        ).lastrowid
    with sqlite3.connect(GENERATED_REPORTS_DB) as connection:
        connection.execute(
            "INSERT INTO generated_reports (patient_report_id, patient_id, generated_text, tone, model_name) VALUES (?, ?, ?, ?, ?)",
            (report_id, patient_id.value.strip(), last_answer, "grounded", model_name),
        )
    set_status(f"Saved answer for patient {patient_id.value.strip()}.")


build_button.on_click(rebuild_index)
ask_button.on_click(ask_question)
clear_button.on_click(clear_ui)
save_button.on_click(save_answer)
display(widgets.HTML("<h2>Medical RAG assistant</h2>"), uploader, build_button, status,
        widgets.HTML("<h3>Ask</h3>"), question,
        widgets.HBox([ask_button, clear_button]),
        widgets.HTML("<h3>Answer</h3>"), answer_output,
        widgets.HTML("<h3>Retrieved sources</h3>"), sources_output,
        widgets.HTML("<h3>Optional persistence</h3>"),
        widgets.HBox([patient_id, save_button]), history_output)
rebuild_index()


HTML(value='<h2>Medical RAG assistant</h2>')

FileUpload(value=(), accept='.txt,.md,.csv,.json,.pdf,.docx', description='Choose files', multiple=True)

Button(button_style='primary', description='Build RAG index', icon='database', style=ButtonStyle())

HTML(value='<b>Status:</b> Ready with built-in documents.')

HTML(value='<h3>Ask</h3>')

Textarea(value='', layout=Layout(height='80px', width='100%'), placeholder='Ask a question about the uploaded …

... (2 outputs left out)

HTML(value='<h3>Retrieved sources</h3>')

Output()

HTML(value='<h3>Optional persistence</h3>')

Output()